# Colab GPU test — video indexing pipeline

Run the cells top to bottom (Runtime → Change runtime type → **GPU**). Order:
1. hardware info → 2. clone + install → 3. data subset → 4. build INT8 and FP16 engines →
5. smoke test on damaged data → 6. ground truth with `reference.py` → 7. accuracy + timing of
INT8 / FP16 / PyTorch → 8. settings sweep → 9. full 50-file run (`results.jsonl.gz`, HARDWARE info).

Needs a Colab **secret** named `HF_TOKEN` (key icon in the left bar) holding a Hugging Face read
token whose account has accepted the FineVideo terms. Colab's GPU (often a T4) is slower than the
graders' RTX 4090, so compare settings *relative* to each other here.

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"   # <-- set this
N_TEST_FILES = 2          # Parquet files for the accuracy / sweep tests (~0.5 GB each)
TEST_VIDEOS = 6           # videos used for the reference comparison
TEST_SECONDS = 60         # seconds per video for the reference comparison
WORK = "/content/work"

## 1. Hardware

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!echo "CPU threads: $(nproc)"; lscpu | grep -E "Model name|^CPU\(s\)"
!free -g | head -2
!df -h /content | tail -1
!df -h /dev/shm | tail -1
!python --version

## 2. Clone and install

In [ ]:
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!git clone -q {REPO_URL} repo || (cd repo && git pull -q)
%cd {WORK}/repo
!sudo apt-get -qq update && sudo apt-get -qq install -y libgl1 libglib2.0-0 > /dev/null
!pip install -q -r requirements.txt
!python -c "import torch, ultralytics, av, tensorrt; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__, '| av', av.__version__, '| tensorrt', tensorrt.__version__)"

## 3. Data subset (test files)

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
pattern = "data/train-0000[" + "".join(str(i) for i in range(N_TEST_FILES)) + "]*"
!hf download HuggingFaceFV/finevideo --repo-type dataset --include "{pattern}" --local-dir {WORK}/finevideo
!ls -la {WORK}/finevideo/data

## 4. Build engines (INT8 and FP16, kept side by side)
`prepare.py` writes `yolo11l.engine` + `yolo11l.json`; we copy each build under its own name.

In [ ]:
!time python prepare.py --precision int8
!cp yolo11l.engine yolo11l_int8.engine && cp yolo11l.json yolo11l_int8.json
!time python prepare.py --precision fp16
!cp yolo11l.engine yolo11l_fp16.engine && cp yolo11l.json yolo11l_fp16.json
!ls -la *.engine *.json

## 5. Smoke test on damaged / unusual inputs
Expect 8 error lines and 20 lines in total, and no crash.

In [ ]:
!python make_broken_data.py --source {WORK}/finevideo/data/train-00000-of-01357.parquet --out {WORK}/testdata
!mkdir -p out
!python run.py --input {WORK}/testdata --output out/test.jsonl --weights yolo11l_int8.engine 2>&1 | grep -E "plan:|device=|BOTTLENECK|Traceback|Error"
!echo "error lines: $(grep -c error out/test.jsonl) (expect 8), total lines: $(wc -l < out/test.jsonl) (expect 20)"

## 6. Ground truth with `reference.py` (plain Ultralytics, every frame, FP32)

In [ ]:
!mkdir -p {WORK}/subset && cp {WORK}/finevideo/data/train-00000-of-01357.parquet {WORK}/subset/
!time python reference.py --input {WORK}/subset --output out/ref.jsonl --max-videos {TEST_VIDEOS} --max-seconds {TEST_SECONDS} 2>&1 | tail -2

## 7. Accuracy and speed: INT8 engine, FP16 engine, PyTorch FP16, and `--gpu-color`

In [ ]:
variants = {
    "int8":           "--weights yolo11l_int8.engine",
    "fp16":           "--weights yolo11l_fp16.engine",
    "pytorch_fp16":   "--weights yolo11l.pt",
    "int8_gpucolor":  "--weights yolo11l_int8.engine --gpu-color",
}
for name, flags in variants.items():
    print(f"\n######## {name}")
    !python run.py --input {WORK}/subset --output out/{name}.jsonl --max-videos {TEST_VIDEOS} --max-seconds {TEST_SECONDS} {flags} --metrics-json out/{name}.json 2>&1 | grep -E "fps overall|real time|model forward|colour convert|BOTTLENECK|Traceback|Error"
    !python evaluate.py --pred out/{name}.jsonl --ref out/ref.jsonl --worst 3 | grep -E "objects F1|people exact|within 1|OVERALL|lowest|seconds="

## 8. Settings sweep (every frame; 12 videos x first 2 minutes per setting, ~2-4 min each on a T4)
Model instances x GPU colour conversion. Pick the fastest setting that passed step 7.

In [ ]:
import json, subprocess, pandas as pd
rows = []
for engine in ["yolo11l_int8.engine", "yolo11l_fp16.engine"]:
    for inst in [1, 2, 3]:
        for gc in [False, True]:
            tag = f"{engine.split('_')[1].split('.')[0]}_i{inst}{'_gc' if gc else ''}"
            cmd = (f"python run.py --input {WORK}/finevideo/data --output out/sweep_{tag}.jsonl --weights {engine} "
                   f"--max-videos 12 --max-seconds 120 "
                   f"--gpu-instances {inst} {'--gpu-color' if gc else ''} --metrics-json out/sweep_{tag}.json")
            subprocess.run(cmd, shell=True, stderr=subprocess.DEVNULL)
            m = json.load(open(f"out/sweep_{tag}.json"))
            rows.append({"setting": tag, "fps": m["frames"] / m["wall"], "x_realtime": m["seconds"] / m["wall"],
                         "wall_s": m["wall"], "model_busy_%": 100 * m["inference_busy"] / (m["wall"] * m["gpu_instances"]),
                         "decode_wait_%": 100 * m["decode"].get("slot_wait", 0) / (m["wall"] * m["workers"]),
                         "gpu_mem_gb": m["gpu_peak_used_gb"]})
            print(rows[-1])
pd.DataFrame(rows).sort_values("fps", ascending=False)

## 9. Full run on all 50 files (needs ~26 GB of disk)
Set `BEST_FLAGS` from steps 7-8. Copy the chosen engine to `yolo11l.engine` so the default
command (the one the graders run) uses it.

In [ ]:
BEST_FLAGS = ""                      # e.g. "--gpu-color" if it passed step 7 and won step 8
CHOSEN_ENGINE = "yolo11l_int8.engine"
!cp {CHOSEN_ENGINE} yolo11l.engine && cp {CHOSEN_ENGINE.replace('.engine', '.json')} yolo11l.json
!hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir {WORK}/finevideo
!ls {WORK}/finevideo/data | wc -l

In [ ]:
!/usr/bin/time -v python run.py --input {WORK}/finevideo/data --output results.jsonl {BEST_FLAGS} --metrics-json out/full_run.json 2> out/full_run.log; tail -60 out/full_run.log
!gzip -kf results.jsonl && ls -la results.jsonl.gz
!echo "lines: $(wc -l < results.jsonl), error lines: $(grep -c error results.jsonl)"

## 10. Details for HARDWARE.md, and download the results

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!lscpu | grep -E "Model name|^CPU\(s\)"; free -g | head -2
!lsblk -d -o NAME,ROTA,SIZE,MODEL | head -5   # ROTA 0 = SSD
!cat /etc/os-release | head -2
!grep -E "Elapsed \(wall clock\)|Maximum resident" out/full_run.log
from google.colab import files
files.download("results.jsonl.gz")
files.download("out/full_run.json")
files.download("out/full_run.log")